# 💊 Pharmacy Sales — EDA, Forecasting & Prediction
نوتبوك بيغطي: تنضيف وفهم الداتا ← تحليل (EDA) ← اختبار ادعاءات الـ Excel dashboard ← Forecasting ← Transaction value prediction ← Product clustering.

**Dataset:** 50,000 transactions · 7 branches · Jan 2022 – Jun 2023 · LKR

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
import ml_core as C

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.titleweight": "bold"})
df = C.load_data()
df.head()

## 1 · Data quality

In [ ]:
print(df.shape)
print("Duplicate IDs:", df.Transaction_ID.duplicated().sum())
print("Date range:", df.Transaction_Date.min().date(), "→", df.Transaction_Date.max().date())
nulls = df.isna().sum(); print(nulls[nulls > 0])
# Line_Total should equal Qty * Price * (1 - Discount)
err = (df.Quantity * df.Unit_Price_LKR * (1 - df.Discount_Rate) - df.Line_Total_LKR).abs().max()
print("Max arithmetic error vs Line_Total:", round(err, 4))

`Strength_mg` فيها nulls لأن الكريمات/الشراب مالهاش mg — دي قيمة طبيعية مش خطأ.

## 2 · KPIs

In [ ]:
kpi = pd.Series({"Revenue (LKR)": df.Line_Total_LKR.sum(), "Transactions": len(df),
                 "Units": df.Quantity.sum(), "Avg order (LKR)": df.Line_Total_LKR.mean(),
                 "Avg discount": df.Discount_Rate.mean()})
kpi.round(3)

## 3 · Exploratory analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
m = df.groupby("YearMonth").Line_Total_LKR.sum() / 1e6
m.plot(marker="o", ax=ax[0], title="Monthly revenue (LKR M)"); ax[0].set_ylim(0, m.max() * 1.15)
b = df.groupby("Branch_Name").Line_Total_LKR.sum().sort_values() / 1e6
b.plot.barh(ax=ax[1], title="Revenue by branch (LKR M, axis from 0)")
plt.tight_layout(); plt.show()
print((b / b.sum()).round(3).sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
c = df.groupby("Medicine_Category").Line_Total_LKR.sum().sort_values() / df.Line_Total_LKR.sum()
c.plot.barh(ax=ax[0], title="Revenue share by category")
a = df.groupby("Age_Group", observed=True).Line_Total_LKR.sum() / 1e6
a.plot.bar(ax=ax[1], title="Revenue by age group (LKR M)", rot=0)
plt.tight_layout(); plt.show()

In [ ]:
heat = df.pivot_table(index="Branch_Name", columns="YearMonth", values="Line_Total_LKR", aggfunc="sum") / 1e6
heat.columns = [c.strftime("%b %y") for c in heat.columns]
plt.figure(figsize=(14, 4)); sns.heatmap(heat, cmap="YlGnBu", annot=True, fmt=".1f", cbar_kws={"label": "LKR M"})
plt.title("Revenue heatmap — branch × month"); plt.show()

## 4 · Fact-checking the dashboard's insights
الـ README قال إن *"Sales volume peaks around 13% discount"*. نتأكد.

In [ ]:
d = df.groupby("Discount_Pct").agg(transactions=("Transaction_ID", "count"), units=("Quantity", "sum"),
                                   avg_order=("Line_Total_LKR", "mean")).reset_index()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=d, x="Discount_Pct", y="transactions", ax=ax[0], color="#0d9488"); ax[0].set_title("Transactions by discount %")
sns.lineplot(data=d, x="Discount_Pct", y="avg_order", marker="o", ax=ax[1]); ax[1].set_title("Average order by discount %")
plt.tight_layout(); plt.show()

mid = d[(d.Discount_Pct > 0) & (d.Discount_Pct < 15)].transactions
print(f"Middle buckets (1–14%): mean={mid.mean():.0f}, std={mid.std():.0f}, 13% = {d.set_index('Discount_Pct').transactions[13]}")
print("13% is", round((d.set_index('Discount_Pct').transactions[13] - mid.mean()) / mid.std(), 2), "std above the mean of the other buckets")

**الخلاصة:** 0% و15% عددهم نص التقريبي (edge buckets)، و13% أعلى شوية بس في حدود الـ noise. **مفيش دليل على إن الخصم بيزوّد الحجم** — والـ revenue بتقل مع زيادة الخصم. الأفضل تكتبيها في الـ dashboard كـ "no clear volume lift from discounts".

In [ ]:
# Is the gap between categories real? Gender share, province share
print(df.groupby("Customer_Gender").Line_Total_LKR.sum().div(df.Line_Total_LKR.sum()).round(4))
print(df.groupby("Province").Line_Total_LKR.sum().div(df.Line_Total_LKR.sum()).round(3).sort_values(ascending=False))

## 5 · Revenue forecasting
Panel: **Branch × Category × Month** (7 × 15 × 18). Train على أول 12 شهر، Test على آخر 6 شهور. بنقارن بـ baseline بسيط (متوسط الشهر) عشان نعرف الموديل بيضيف إيه فعلاً.

In [ ]:
panel = C.build_monthly_panel(df)
cut = panel.t.max() - 5
train, test = panel[panel.t < cut], panel[panel.t >= cut]
feats = C.FORECAST_CAT + C.FORECAST_NUM

def make(kind):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), C.FORECAST_CAT),
                             ("num", StandardScaler(), C.FORECAST_NUM)])
    est = Ridge(alpha=1.0) if kind == "ridge" else HistGradientBoostingRegressor(max_depth=4, learning_rate=.05, max_iter=300, random_state=42)
    return Pipeline([("pre", pre), ("model", est)])

base = train.groupby(C.FORECAST_CAT).Revenue.mean().rename("p").reset_index()
bp = test.merge(base, on=C.FORECAST_CAT).p
rows = {"baseline": (mean_absolute_error(test.Revenue, bp), r2_score(test.Revenue, bp))}
preds = {}
for k in ["ridge", "gbm"]:
    mdl = make(k).fit(train[feats], train.Revenue); preds[k] = mdl.predict(test[feats])
    rows[k] = (mean_absolute_error(test.Revenue, preds[k]), r2_score(test.Revenue, preds[k]))
pd.DataFrame(rows, index=["MAE", "R2"]).T.round(3)

الموديلات قريبة جدًا من الـ baseline — لأن الداتا ثابتة (مفيش seasonality قوية أو trend واضح). ده طبيعي في داتا synthetic، وأحسن من إننا نبالغ في الدقة.

In [ ]:
# Total-level view of the hold-out
t = test.assign(pred=preds["ridge"]).groupby("YearMonth")[["Revenue", "pred"]].sum() / 1e6
t.plot(marker="o", title="Hold-out: actual vs Ridge (total revenue, LKR M)"); plt.ylim(0, t.values.max() * 1.2); plt.show()
print("Total-level MAPE:", round((abs(t.Revenue - t.pred) / t.Revenue).mean() * 100, 2), "%")

## 6 · Transaction value prediction

In [ ]:
catalog = C.medicine_catalog(df)
d = C.add_avg_price(df, catalog)
X, y = d[C.VALUE_CAT + C.VALUE_NUM], d.Line_Total_LKR
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=42)
vm = Pipeline([("pre", ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), C.VALUE_CAT)], remainder="passthrough")),
               ("model", HistGradientBoostingRegressor(max_depth=6, learning_rate=.08, max_iter=400, random_state=42))]).fit(Xtr, ytr)
p = vm.predict(Xte)
print("MAE:", round(mean_absolute_error(yte, p), 1), "| R2:", round(r2_score(yte, p), 4))
formula = Xte.Quantity * Xte.Avg_Price * (1 - Xte.Discount_Rate)
print("Pure formula (qty × avg price × (1-disc)) R2:", round(r2_score(yte, formula), 4))
plt.scatter(yte[:2000], p[:2000], s=6, alpha=.4); plt.plot([0, 20000], [0, 20000], "r--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Predicted vs actual transaction value"); plt.show()

**ملاحظة مهمة:** الـ R² العالي ناتج أساسًا من المعادلة (كمية × سعر × (1 − خصم))، مش من "ذكاء" الموديل. الـ ML بيفيد هنا لو ضفتي أعمدة بتأثر فعلاً (موسم، عروض، مخزون…).

In [ ]:
# What drives the prediction?  (permutation importance)
from sklearn.inspection import permutation_importance
r = permutation_importance(vm, Xte.sample(5000, random_state=1), yte.loc[Xte.sample(5000, random_state=1).index], n_repeats=3, random_state=0)
imp = pd.Series(r.importances_mean, index=list(X.columns)).sort_values()
plt.barh(imp.index.tolist(), imp.values); plt.title("Permutation importance"); plt.show()

## 7 · Product clustering (KMeans)

In [ ]:
prod = df.groupby("Medicine_ID").agg(price=("Unit_Price_LKR", "mean"), units=("Quantity", "sum"),
                                      revenue=("Line_Total_LKR", "sum")).reset_index()
Z = StandardScaler().fit_transform(np.log1p(prod[["price", "units", "revenue"]]))
inertia = [KMeans(k, n_init=10, random_state=0).fit(Z).inertia_ for k in range(2, 8)]
plt.plot(range(2, 8), inertia, marker="o"); plt.title("Elbow"); plt.xlabel("k"); plt.show()

In [ ]:
prod["cluster"] = KMeans(4, n_init=10, random_state=0).fit_predict(Z)
prod = prod.merge(catalog[["Medicine_ID", "Medicine_Name", "Medicine_Category"]], on="Medicine_ID")
print(prod.groupby("cluster")[["price", "units", "revenue"]].mean().round(0).assign(n=prod.cluster.value_counts().sort_index()))
sns.scatterplot(data=prod, x="price", y="units", hue="cluster", palette="Set2", s=70); plt.title("Medicine segments: price vs units"); plt.show()

## 8 · Next steps
- ضيفي بيانات حقيقية (مخزون، عروض، مواسم/أمراض موسمية) عشان الـ forecasting يتحسن فعلًا.
- لو في customer_id: RFM + churn prediction.
- شغّلي `python train_models.py` لتحديث الموديلات، وبعدها `streamlit run app_streamlit.py` أو `python flask_app/app.py`.